# VQCC verified portfolio: a three-task assembly at 24.9+

**What is submitted (2026-11-06).** A best-of-public portfolio: each member is the
cheapest circuit that passes the competition's own verifier unmodified, chosen out of
every circuit published for this competition to date. The component digests and cost
model are pinned below; the assembled `submission.zip` is written at the end.

| task | circuit | cost | ratio | made by |
|---|---|---|---|---|
| invsqrt | `c17971af` | 97,746.90 | 21.25x | yoyobar's 62-qubit line (Ehan Mallick merges, community edits) |
| dqi | `768495b4` | 374,767,710,994 | 65.21x | daifanhao's four exact retained-product chains on syed3000's V4 decoder (891q) |
| dfthcsos | `a71c1664` + 15 loaders | 299,671,05x | 11.20x | heliosli's W/H2_I weights on syed3000's 8x143x44 table and loaders |

Geometric mean of the ratios: **24.94** at the default salt. None of the shipped
circuits is ours; the work here is the selection and verification machinery, the
negative results below, and the factorization program described in section 5.


## What we tried that did not work

Every row is a measured run, not a guess. Failed candidates are kept in the repo under
`rejected_` names; the numbers below are from the official verifiers or the
factorization metric code.

| idea | what we measured | result |
|---|---|---|
| Continuous W-refit then progressive requantize then lambda optimization on the R8x143x62 fit (our full chain, ~2 days of compute) | continuous error_2 13.68 mHa -> quantized 18.39 -> final lambda_eff 47.53, est. cost ~341M vs shipped ~300M | Worse. Rejected. |
| C-truncation of the same fit to C50, then refit and requantize | post-truncation refit reached error_2 17.13 mHa, but progressive quantization climbed to 21.08 mHa at stage 30/36 | Over the 20 mHa gate. Dead. |
| Same truncation to C44 | error_2 ~31 mHa immediately after refit | Dead earlier. |
| Dropping a merged group (R8 -> R7) on the SomeJoint fit | error_2 jumps to ~1,600 mHa before refit | Closed. |
| Our own Berlekamp-Massey + Chien + phase DQI decoder, end-to-end | phase verified correct on all in-range cases; workspace cleanup leaves ~1,300 qubits of replay garbage; est. 2-4x ratio | Not competitive; kept as research. |
| mad-dqi source-tree rebuilds (hankel_half, cyclic_gold, gold_extraction configs) | 881-899 qubits but 965k-1.43M static Toffolis | All worse than the deployed 891q/397k circuits. |
| v40 DQI submission with 1 value error in 102,400 shots | nominal cost slightly lower, but the hidden-salt repetition penalty dominates | LB went DOWN. Lesson: only zero-error decoders ship. |
| Merge+shrink factorization ladder reproduced at default fit iterations | error_2 24.1 mHa at (9,143,46) vs the author's published 13.51 | Undertrained; rerun at 12k-20k iters (section 5). |


## 1. The scoring model

`score_task = lambda_eff x T_total x Q_total` per task; the leaderboard score is the
geometric mean of `baseline_cost / our_cost` over the three tasks. Multiplicative
gains compound, so the cheapest verified circuit for each task goes in, whoever made it.

- **invsqrt** (baseline 2,076,774): cost = C x Q. 62 qubits is the published floor;
  recent gains are exact gate-level edits (degree-2 CCZ to Clifford, ladder rebases)
  worth ~1 Toffoli-equivalent each.
- **dqi** (baseline ~2.44e13): cost scales with Q^2 x T plus an error-rate repetition
  factor. Every qubit cut is worth ~1.5x a Toffoli cut. The frontier (889 -> 891
  qubits, ~397k mean T) is now fought over tens of Toffolis by byte-level splices.
- **dfthcsos** (baseline 3.3556e9): lambda_eff x T x Q. T = angle-word loaders
  (~4.2k T/pass x 2) + state prep (~4.4k, scales with C) + Givens (~2.25k). The
  factorization's word count D = R x B + 76 and its lambda_eff are the structural
  levers; the current frontier shape is 8x143x44 (1,220 words, lambda_eff ~46.9).


## 2. The verified portfolio

Every candidate circuit is run through the competition's official verifier locally
before it is allowed in. A circuit that passes but shows any value or phase errors
is treated as suspect: the hidden half of the leaderboard re-samples shots, and our
v40 submission proved a single measured error can cost more than the nominal saving.
The manifest below pins every file by SHA-256.


In [ ]:
import hashlib, json, zipfile, os, shutil
from pathlib import Path
import glob

# locate the dataset files wherever they mounted: any angle loader under /kaggle/input
cands = sorted(glob.glob('/kaggle/input/**/dfthcsos__submission.npz', recursive=True))
assert cands, 'dataset files not found: ' + str(sorted(glob.glob('/kaggle/input/*/*'))[:30])
ROOT = Path(cands[0]).parent
print('dataset root:', ROOT)
OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/tmp/nb_out')
OUT.mkdir(exist_ok=True)

files = sorted(p for p in ROOT.iterdir() if p.suffix in ('.kmb', '.npz'))
MANIFEST = {}
for p in files:
    MANIFEST[p.name] = hashlib.sha256(p.read_bytes()).hexdigest()
print(f'{len(files)} files hashed')
for k, v in sorted(MANIFEST.items()):
    print(f'  {v[:12]}  {k}')



In [ ]:
# assemble submission.zip: invsqrt.kmb + dqi.kmb + dfthcsos/{angle_k_*,submission.npz}
sub = OUT / 'submission'
shutil.rmtree(sub, ignore_errors=True)
(sub / 'dfthcsos').mkdir(parents=True)
for p in files:
    name = p.name
    if name.startswith('dfthcsos__'):
        shutil.copyfile(p, sub / 'dfthcsos' / name[len('dfthcsos__'):])
    elif name in ('invsqrt.kmb', 'dqi.kmb'):
        shutil.copyfile(p, sub / name)
with zipfile.ZipFile(OUT / 'submission.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(sub.rglob('*')):
        z.write(p, p.relative_to(sub))
print('wrote', OUT / 'submission.zip')
print(sorted(str(p.relative_to(sub)) for p in sub.rglob('*')))


## 3. How the components are chosen

New public releases are scanned continuously: every dataset, notebook output and
curation manifest published for this competition. For each task we keep the cheapest
artifact that (a) passes the official verifier unmodified and (b) shows zero value
or phase errors across the verifier's shots. Mixing is per-task: the three members
of the current zip come from three different authors' release lines.

The verified-cost ordering, not stated claims, decides: several published circuits
are advertised with better numbers than they verify at, and two were dropped from
our manifest for that reason.


## 4. Verification records

The records shipped in this dataset are the outputs of the competition's own
verifiers, run locally on the exact bytes hashed in section 2.


In [ ]:
for rec in ('invsqrt.verify.json', 'dqi.verify.json', 'dfthcsos.verify.json'):
    p = ROOT / rec
    if p.exists():
        d = json.loads(p.read_text())
        keys = ('status', 'score', 'num_qubits', 'toffoli_count', 'lambda_eff',
                'value_errors', 'phase_errors', 'worst_error_over_tolerance')
        print(rec, {k: d[k] for k in keys if k in d})



## 5. Own work: the factorization program

The remaining public headroom is in the chemistry factorization itself. We run the
published merge-then-shrink pipeline (`marculera/dfthc-merge-pipeline`, Apache-2.0):
DF-THC groups are merged exactly (block-diagonal), then faded and retrained down a
shape ladder with float64 L-BFGS. The public seed (9,180,90) starts at error_2
6.38 mHa and the published ladder reaches (9,143,46) at 13.51 mHa continuous.

Our open question is whether a B-biased ladder reaches territory nobody has
published - (9,137,42), (9,131,41), or (9,129,40), with B=129 the last value before
the verifier's temporary-qubit cliff - with continuous error_2 still under the ~13.5
mHa needed for 8-bit quantization. A first reproduction at default fit iterations
landed at 24.1 mHa (see the nulls table); the corrected run is in progress and its
checkpoints are logged in the repository's `AGENTS.md`.

If a new-shape fit lands inside the error gates, it becomes ours in the way that
matters here: a factorization at parameters nobody else has verified, refit and
compiled with public tools, with every step logged.


## 6. Provenance and license

Components are assembled from Apache-2.0 public releases by the community. Per-file
digests in section 2 identify exactly which artifacts ship. Credit chains, as stated
by the authors' own releases:

- **invsqrt** `c17971af`: yoyobar's 62-qubit family; Ehan Mallick merges; somejoint
  windows/normalization/phase split; exact edits by Isabe11aM00re, leotang09,
  qiuqiuh, kirilltym and others in that line.
- **dqi** `768495b4`: daifanhao's retained-product chains (busyaprime's idea,
  Atakan Aldemir's scanner/emitter) on syed3000's V4 decoder; upstream SomeJoint
  (Agniv Banerjee) / yoyobar / heliosli decoder lineage.
- **dfthcsos**: heliosli's W/H2_I weights on syed3000's 8x143x44 factorization and
  15 angle loaders; fit lineage marculera (merge pipeline) / Atakan Aldemir /
  manwithacat (dfthc_refit) / Isabe11aM00re.

The value added here is verification discipline, honest bookkeeping of what does
and does not work, and the factorization program in section 5.


## Changelog

| version | score | change |
|---|---|---|
| v1 | 22.68 | First verified portfolio (public picks of 2026-10-28). |
| v2 | 23.42 | Adopted hedge-wizardry curation (HW-04 chem, 62q invsqrt, 888q dqi). |
| v3 | 24.25 | HW-09 dqi (891q splice line) + HW-15 chem (313.7M). |
| v4 | 24.54 | syed3000's own portfolio beats all three picks. |
| v5 | 24.59 | alexchilton's 24.59 picks (heliosli chem 311.5M). |
| v6 | 24.86 | Best-of: malachi invsqrt + HW-22 dqi + HW-24 chem (301.9M). |
| v7 | 24.94 | daifanhao dqi chains + heliosli weights on syed's 8x143x44 table. |
